<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two findings and methodology questions

**Finding 1: The capstone paper states that a tree-based model provides a more informative prioritization signal than a heuristic baseline.**

The label is whether a content item’s observed March CTR is below the median CTR for its March search-position bucket. This is a position-adjusted observational proxy for a CTR review opportunity, not a record of whether an optimization produced more clicks.

**Methodology question:** Does the grouped-client validation support the claim?  
**Constructive assessment:** It supports a narrower claim: on held-out pseudonymized clients, the model ranked the defined March proxy better than the February-only baseline. It does not validate causal impact from an editorial or SEO change.

**Finding 2: The paper states that grouped validation reduces leakage across related content entities.**

**Methodology question:** Is the evaluation fully deployment-like?  
**Constructive assessment:** Holding out whole clients prevents the model from learning the same client in both training and validation. However, the modelling population requires March search availability and at least 100 March impressions. That outcome-window eligibility choice should be disclosed, because a fully future deployment cohort would not yet know its March eligibility.

In [6]:
import pandas as pd

paper_claim_audit = pd.DataFrame(
    [
        {
            "paper_finding": (
                "Tree-based model provides a more informative prioritization "
                "signal than the heuristic baseline."
            ),
            "label_source": (
                "Observed March CTR below the median CTR of the item's "
                "March position bucket."
            ),
            "validation_question": (
                "Does grouped client holdout support generalization to "
                "unseen clients?"
            ),
            "honest_boundary": (
                "Supports held-out ranking of the proxy label, not causal "
                "CTR improvement after an edit."
            ),
        },
        {
            "paper_finding": (
                "Grouped validation reduces leakage across related content entities."
            ),
            "label_source": (
                "Same March position-adjusted CTR proxy."
            ),
            "validation_question": (
                "Does the population definition depend on the March outcome window?"
            ),
            "honest_boundary": (
                "March availability and 100-impression eligibility are "
                "outcome-window selection choices and must be disclosed."
            ),
        },
    ]
)

print("=== Research claim audit ===")
display(paper_claim_audit)


=== Research claim audit ===


,paper_finding,label_source,validation_question,honest_boundary
0,Tree-based model provides a more informative p...,Observed March CTR below the median CTR of the...,Does grouped client holdout support generaliza...,"Supports held-out ranking of the proxy label, ..."
1,Grouped validation reduces leakage across rela...,Same March position-adjusted CTR proxy.,Does the population definition depend on the M...,March availability and 100-impression eligibil...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Random-row versus grouped-client validation

I will evaluate the same February-to-March Random Forest under two designs.

The random-row split is intentionally weaker: it can place content from the same client in both training and validation. The grouped-client split holds out entire pseudonymized clients, matching the honest ML-08 design. A difference between these results indicates how much client-specific structure may affect a random-row evaluation.

The target is the same March position-adjusted CTR proxy used in ML-08. March eligibility is an outcome-window population choice and is disclosed in the limitations; it is not used as a predictor.

In [7]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import (
    StratifiedGroupKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{hf_token}')"
)

FEB = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/*.parquet"
)

MARCH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

model_df = con.execute(f"""
    WITH february_features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS feb_impressions,
            AVG(NULLIF(gsc_avg_position, 0)) AS feb_avg_position,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN scroll_events END)
                AS feb_scroll_events,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_social END)
                AS feb_sessions_social,
            COUNT(*) AS feb_observed_days,
            MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
                AS has_ga4_data
        FROM read_parquet('{FEB}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    march_outcomes AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            SUM(gsc_clicks)::DOUBLE
                / NULLIF(SUM(gsc_impressions), 0) AS march_ctr,
            AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position
        FROM read_parquet('{MARCH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    march_bucketed AS (
        SELECT *,
            CASE
                WHEN march_avg_position <= 3 THEN '1-3'
                WHEN march_avg_position <= 10 THEN '4-10'
                WHEN march_avg_position <= 20 THEN '11-20'
                WHEN march_avg_position <= 50 THEN '21-50'
                ELSE '51+'
            END AS march_position_bucket
        FROM march_outcomes
        WHERE march_impressions >= 100
          AND march_avg_position IS NOT NULL
    ),
    labelled_march AS (
        SELECT *,
            MEDIAN(march_ctr) OVER (
                PARTITION BY march_position_bucket
            ) AS march_bucket_median_ctr
        FROM march_bucketed
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.feb_impressions,
        f.feb_avg_position,
        f.feb_scroll_events,
        f.feb_sessions_social,
        f.feb_observed_days,
        f.has_ga4_data,
        CASE
            WHEN f.feb_avg_position <= 3 THEN '1-3'
            WHEN f.feb_avg_position <= 10 THEN '4-10'
            WHEN f.feb_avg_position <= 20 THEN '11-20'
            WHEN f.feb_avg_position <= 50 THEN '21-50'
            ELSE 'missing_or_51+'
        END AS feb_position_bucket,
        CASE
            WHEN m.march_ctr < m.march_bucket_median_ctr THEN 1
            ELSE 0
        END AS target
    FROM february_features AS f
    INNER JOIN labelled_march AS m
        USING (client_hash_id, content_hash_id)
    WHERE f.feb_impressions >= 100
      AND f.feb_avg_position IS NOT NULL
""").df()

feature_columns = [
    "feb_impressions",
    "feb_avg_position",
    "feb_scroll_events",
    "feb_sessions_social",
    "feb_observed_days",
    "has_ga4_data",
    "feb_position_bucket",
]

X = model_df[feature_columns].copy()
y = model_df["target"].copy()
groups = model_df["client_hash_id"].copy()

numeric_features = [
    "feb_impressions",
    "feb_avg_position",
    "feb_scroll_events",
    "feb_sessions_social",
    "feb_observed_days",
    "has_ga4_data",
]

def make_model():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        (
                            "impute",
                            SimpleImputer(
                                strategy="median",
                                add_indicator=True,
                            ),
                        ),
                    ]
                ),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        ("encode", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                ["feb_position_bucket"],
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "random_forest",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=12,
                    min_samples_leaf=20,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )

def precision_at_k(y_true, scores, k=20):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()

def ranking_metrics(y_true, scores):
    return {
        "Precision@20": precision_at_k(y_true, scores, k=20),
        "Average Precision": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
    }

# 1. Intentionally weaker random-row split
X_train_random, X_val_random, y_train_random, y_val_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

random_model = make_model()
random_model.fit(X_train_random, y_train_random)
random_scores = random_model.predict_proba(X_val_random)[:, 1]

random_client_overlap = len(
    set(groups.loc[X_train_random.index])
    .intersection(set(groups.loc[X_val_random.index]))
)

# 2. Honest grouped-client split
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

candidate_splits = list(splitter.split(X, y, groups))
overall_rate = y.mean()

group_train_index, group_val_index = min(
    candidate_splits,
    key=lambda split: abs(y.iloc[split[1]].mean() - overall_rate),
)

X_train_group = X.iloc[group_train_index]
X_val_group = X.iloc[group_val_index]
y_train_group = y.iloc[group_train_index]
y_val_group = y.iloc[group_val_index]

group_model = make_model()
group_model.fit(X_train_group, y_train_group)
group_scores = group_model.predict_proba(X_val_group)[:, 1]

grouped_client_overlap = len(
    set(groups.iloc[group_train_index])
    .intersection(set(groups.iloc[group_val_index]))
)

random_results = ranking_metrics(y_val_random, random_scores)
grouped_results = ranking_metrics(y_val_group, group_scores)

comparison = pd.DataFrame(
    {
        "metric": list(random_results.keys()),
        "random_row_split": list(random_results.values()),
        "grouped_client_holdout": list(grouped_results.values()),
    }
)

print("=== Validation populations ===")
print(f"Rows: {len(model_df):,}")
print(f"Overall target rate: {overall_rate:.3f}")
print(f"Random-split client overlap: {random_client_overlap}")
print(f"Grouped-holdout client overlap: {grouped_client_overlap}")
print(f"Grouped validation target rate: {y_val_group.mean():.3f}")

print("\n=== Random-row versus grouped-client validation ===")
display(comparison.round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Validation populations ===
Rows: 73,481
Overall target rate: 0.388
Random-split client overlap: 32
Grouped-holdout client overlap: 0
Grouped validation target rate: 0.372

=== Random-row versus grouped-client validation ===


,metric,random_row_split,grouped_client_holdout
0,Precision@20,0.650,0.850
1,Average Precision,0.570,0.583
2,ROC-AUC,0.722,0.730


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Final feature leakage audit

The final model uses only February features: impressions, average position, engagement signals, observation coverage, GA4 availability, and a February position bucket. The target is derived only from March CTR and March position buckets, so it is not available to the model at prediction time.

No client or content identifiers, March outcome fields, product flags, previous action scores, reason codes, or future-window features are included in the model inputs. The population uses March availability and impression eligibility; this outcome-window selection choice is disclosed as a limitation rather than treated as a predictor.

In [8]:
from sklearn.tree import DecisionTreeClassifier

final_score_inputs = feature_columns.copy()

forbidden_terms = [
    "march",
    "target",
    "label",
    "future",
    "next",
    "action",
    "reason",
    "score",
    "product_flag",
    "client_hash",
    "content_hash",
]

suspect_inputs = [
    field
    for field in final_score_inputs
    if any(term in field.lower() for term in forbidden_terms)
]

feature_timeline = pd.DataFrame(
    {
        "field_group": [
            "Final model features",
            "Target",
            "Identifiers",
            "Product outputs",
        ],
        "timing_or_role": [
            "February only; available before the March outcome window",
            "March CTR relative to March position-bucket median",
            "Used only for grouping and split construction, never as features",
            "Not queried or used as features",
        ],
    }
)

print("=== Final feature-set audit ===")
print("Final score inputs:", final_score_inputs)
print("Suspect input names:", suspect_inputs)
print("Target present in X:", "target" in X.columns)
print(
    "Identifier present in X:",
    any(name in X.columns for name in ["client_hash_id", "content_hash_id"]),
)

assert not suspect_inputs
assert "target" not in X.columns
assert "client_hash_id" not in X.columns
assert "content_hash_id" not in X.columns

display(feature_timeline)

# Deliberate leakage harness:
# A copied target should create an unrealistically perfect result.
# It is never included in the honest final model.
leak_train = pd.DataFrame(
    {"leaked_target_copy": np.asarray(y_train_group)}
)
leak_validation = pd.DataFrame(
    {"leaked_target_copy": np.asarray(y_val_group)}
)

leak_demo_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=1,
)
leak_demo_model.fit(leak_train, y_train_group)
leak_demo_scores = leak_demo_model.predict_proba(leak_validation)[:, 1]

print("\n=== Deliberate label-leakage harness ===")
print(
    f"Honest grouped-model ROC-AUC: "
    f"{roc_auc_score(y_val_group, group_scores):.3f}"
)
print(
    f"Deliberately leaked ROC-AUC: "
    f"{roc_auc_score(y_val_group, leak_demo_scores):.3f}"
)
print(
    f"Deliberately leaked Precision@20: "
    f"{precision_at_k(y_val_group, leak_demo_scores, k=20):.3f}"
)
print(
    "The copied label is used only for this audit demonstration and is "
    "not part of final_score_inputs."
)

=== Final feature-set audit ===
Final score inputs: ['feb_impressions', 'feb_avg_position', 'feb_scroll_events', 'feb_sessions_social', 'feb_observed_days', 'has_ga4_data', 'feb_position_bucket']
Suspect input names: []
Target present in X: False
Identifier present in X: False


,field_group,timing_or_role
0,Final model features,February only; available before the March outc...
1,Target,March CTR relative to March position-bucket me...
2,Identifiers,"Used only for grouping and split construction,..."
3,Product outputs,Not queried or used as features



=== Deliberate label-leakage harness ===
Honest grouped-model ROC-AUC: 0.730
Deliberately leaked ROC-AUC: 1.000
Deliberately leaked Precision@20: 1.000
The copied label is used only for this audit demonstration and is not part of final_score_inputs.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite and error inspection

**Too strong:** “The model identifies pages that will benefit from optimization.”

**Public-safe rewrite:** “Using February signals, the Random Forest ranked observed March below-benchmark CTR items ahead of a transparent February-only baseline on the held-out pseudonymized clients. This is a measured, directional decision-support result for the defined proxy label, not proof that an optimization will increase clicks.”

The grouped validation result is limited to this dataset, feature set, target definition, and March eligibility population. It should not be interpreted as a claim about Google’s algorithm or causal ranking improvement.

In [9]:
# Public-safe claim audit and top-20 error inspection

grouped_precision_at_20 = precision_at_k(
    y_val_group,
    group_scores,
    k=20,
)

grouped_average_precision = average_precision_score(
    y_val_group,
    group_scores,
)

grouped_roc_auc = roc_auc_score(
    y_val_group,
    group_scores,
)

claim_language_audit = pd.DataFrame(
    {
        "claim_version": [
            "Too strong",
            "Public-safe",
        ],
        "claim": [
            (
                "The model identifies pages that will benefit from "
                "optimization."
            ),
            (
                "On held-out pseudonymized clients, a February-feature "
                "Random Forest ranked the defined March CTR proxy above "
                "a transparent baseline. This is directional "
                "decision-support, not causal proof of click improvement."
            ),
        ],
    }
)

print("=== Claim language audit ===")
display(claim_language_audit)

claim_metrics = pd.DataFrame(
    {
        "grouped_validation_measure": [
            "Precision@20",
            "Average Precision",
            "ROC-AUC",
        ],
        "value": [
            grouped_precision_at_20,
            grouped_average_precision,
            grouped_roc_auc,
        ],
    }
)

print("=== Evidence for the safe claim ===")
display(claim_metrics.round(3))

grouped_review = X_val_group[
    [
        "feb_impressions",
        "feb_avg_position",
        "feb_scroll_events",
        "feb_sessions_social",
        "feb_observed_days",
        "has_ga4_data",
    ]
].copy()

grouped_review["actual_below_benchmark"] = np.asarray(y_val_group)
grouped_review["model_score"] = group_scores

top_20_grouped = grouped_review.sort_values(
    "model_score",
    ascending=False,
).head(20)

false_positive_examples = top_20_grouped[
    top_20_grouped["actual_below_benchmark"] == 0
]

print("=== Top-20 review-cutoff errors ===")
print(f"Top-20 false positives: {len(false_positive_examples)}")

if len(false_positive_examples) > 0:
    print(
        "These are observed high-score candidates whose March CTR was not "
        "below its position-bucket benchmark. They may reflect missing "
        "signals, query mix, or normal variation."
    )
    display(false_positive_examples.round(3))
else:
    print("No false positives occurred in this particular top-20 queue.")

=== Claim language audit ===


,claim_version,claim
0,Too strong,The model identifies pages that will benefit f...
1,Public-safe,"On held-out pseudonymized clients, a February-..."


=== Evidence for the safe claim ===


,grouped_validation_measure,value
0,Precision@20,0.850
1,Average Precision,0.583
2,ROC-AUC,0.730


=== Top-20 review-cutoff errors ===
Top-20 false positives: 3
These are observed high-score candidates whose March CTR was not below its position-bucket benchmark. They may reflect missing signals, query mix, or normal variation.


,feb_impressions,feb_avg_position,feb_scroll_events,feb_sessions_social,feb_observed_days,has_ga4_data,actual_below_benchmark,model_score
69370,101.0,7.178,NaN,NaN,10,0,0,0.778
66869,115.0,7.298,NaN,NaN,10,0,0,0.776
46893,133.0,5.323,NaN,NaN,26,0,0,0.776


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

* [x] Every section above is filled — markdown thinking AND the code that backs it
* [x] The notebook runs top to bottom with no errors (Runtime → Run all)
* [x] No client names, URLs, or private queries anywhere
* [x] My claims use careful words: observed, measured, directional, decision-support
* [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.